In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np
import librosa
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load the labels CSV file
labels_df = pd.read_csv('/content/iemocap_full_dataset.csv')

In [ ]:
# Specify the path to the zipped audio folder in Google Drive
zip_path = '/content/drive/MyDrive/Copy of IEMOCAP_new.zip'  # Update with your actual zip path

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/IEMOCAP_extracted')

In [ ]:
# Define paths to train, validation, and test folders after extraction
train_path = '/content/IEMOCAP_extracted/Copy of IEMOCAP/IEMOCAP_Copied/train'
val_path = '/content/IEMOCAP_extracted/Copy of IEMOCAP/IEMOCAP_Copied/validate'
test_path = '/content/IEMOCAP_extracted/Copy of IEMOCAP/IEMOCAP_Copied/test'


# Function to extract frame-level energy and pitch features from an audio file
def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

    # Calculate energy per frame (RMS energy)
    energy = librosa.feature.rms(y=y)[0]  # RMS energy returns a vector for each frame

    # Calculate pitch per frame using librosa's pitch tracking
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch = []

    # Extract the highest magnitude pitch for each frame
    for i in range(pitches.shape[1]):
        index = magnitudes[:, i].argmax()
        pitch_value = pitches[index, i]
        pitch.append(pitch_value if pitch_value > 0 else np.nan)  # Use NaN for unvoiced frames

    return energy, np.array(pitch)

# Process files and save features in separate CSV files for train, val, and test
def process_and_save(folder_path, labels_df, output_csv):
    features_list = []

    # Process only the first 5 rows in the dataset for testing purposes
    for _, row in labels_df.head(10039).iterrows():  # Limit to first 5 for testing
        file_name = row['path'].replace('/', '_')  # Replace slashes with underscores
        label = row['emotion']  # Use the 'emotion' column for labels

        # Check if the file exists in the folder
        file_path = os.path.join(folder_path, file_name)

        if not os.path.isfile(file_path):
            print(f"File '{file_name}' not found in folder '{folder_path}', skipping.")
            continue  # Skip if file does not exist in the current folder

        # Extract features if file is found
        energy, pitch = extract_features(file_path)

        if energy is None or pitch is None:  # Skip if an error occurred during feature extraction
            print(f"Skipping file due to load error: {file_name}")
            continue

        features_list.append({
            'file_name': file_name,
            'emotion': label,
            'energy_values': energy,  # Frame-level energy vector
            'energy_shape': energy.shape,  # Shape of the energy vector
            'pitch_values': pitch,         # Frame-level pitch vector
            'pitch_shape': pitch.shape     # Shape of the pitch vector
        })

    # Remove existing output file if it exists
    if os.path.isfile(output_csv):
        os.remove(output_csv)

    # Save to CSV
    features_df = pd.DataFrame(features_list)
    features_df.to_csv(output_csv, index=False)

# Run for train, val, and test folders, limiting to the first 5 rows only
process_and_save(train_path, labels_df, '/content/train_features.csv')
process_and_save(val_path, labels_df, '/content/val_features.csv')
process_and_save(test_path, labels_df, '/content/test_features.csv')



Streaming output truncated to the last 5000 lines.
File 'Session3_sentences_wav_Ses03F_script02_1_Ses03F_script02_1_F008.wav' not found in folder '/content/IEMOCAP_extracted/Copy of IEMOCAP/IEMOCAP_Copied/test', skipping.
File 'Session3_sentences_wav_Ses03F_script02_1_Ses03F_script02_1_F009.wav' not found in folder '/content/IEMOCAP_extracted/Copy of IEMOCAP/IEMOCAP_Copied/test', skipping.
File 'Session3_sentences_wav_Ses03F_script02_1_Ses03F_script02_1_F010.wav' not found in folder '/content/IEMOCAP_extracted/Copy of IEMOCAP/IEMOCAP_Copied/test', skipping.
File 'Session3_sentences_wav_Ses03F_script02_1_Ses03F_script02_1_F011.wav' not found in folder '/content/IEMOCAP_extracted/Copy of IEMOCAP/IEMOCAP_Copied/test', skipping.
File 'Session3_sentences_wav_Ses03F_script02_1_Ses03F_script02_1_F013.wav' not found in folder '/content/IEMOCAP_extracted/Copy of IEMOCAP/IEMOCAP_Copied/test', skipping.
File 'Session3_sentences_wav_Ses03F_script02_1_Ses03F_script02_1_F014.wav' not found in folder